# Fetch comments — top comments of food review videos (daily job, after refresh)

Until 2026-10-31 comments were only fetched when someone clicked in the app, so `silver.stg_comments` was almost empty.

- **Which videos:** `gold.dim_video` rows classified as food reviews (`video_type` set and `is_food_review`), with `comment_count > 0` in Postgres, never fetched or fetched more than `comments_refetch_days` (30) days ago; never fetched first, then most viewed.
- **API:** one `commentThreads.list` per video (`order = relevance`, `maxResults = 100`) = **1 quota unit**; at most `comments_max_videos` (200) per run, so ≤ 200 of the 10,000 free units per day.
- **Writes:** `public.comments` (UPSERT, `source = fetch_comments`) and `public.comment_fetches` (one row per video: last fetch, count, status ok / disabled / not_found). Bronze → `stg_comments` → `gold.fact_comment` pick them up in the same run.

The task never fails the job: a quota error stops the loop, errors are printed, and the daily run goes on.

In [ ]:
%pip install --quiet "psycopg>=3.2.0" "httpx>=0.28.0" "python-dotenv>=1.0.0"


In [ ]:
# --- Config ---
import os
import sys

for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
GOLD = dbutils.widgets.get("gold")
dbutils.widgets.text("comments_max_videos", "200")
dbutils.widgets.text("comments_refetch_days", "30")
dbutils.widgets.dropdown("dry_run", "false", ["false", "true"])

MAX_VIDEOS = int(dbutils.widgets.get("comments_max_videos"))
REFETCH_DAYS = int(dbutils.widgets.get("comments_refetch_days"))
DRY_RUN = dbutils.widgets.get("dry_run") == "true"

os.environ.update({
    "PGHOST": "ep-still-butterfly-d8fd3r70.database.us-east-2.cloud.databricks.com",
    "PGDATABASE": "databricks_postgres",
    "PGUSER": "admin",
    "PGPORT": "5432",
    "PGSSLMODE": "require",
    "APP_SCHEMA": "public",
    "PGPASSWORD": dbutils.secrets.get("youtube", "pg_password"),
    "YOUTUBE_API_KEY": dbutils.secrets.get("youtube", "api_key"),
    # FIPS: use psycopg's pure-Python implementation, not the bundled OpenSSL of psycopg[binary]
    "PSYCOPG_IMPL": "python",
})

notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
app_dir = os.path.normpath(f"/Workspace{notebook_dir}/../../app")  # src/notebooks/refresh -> src/app
sys.path.insert(0, app_dir)
print(f"App code: {app_dir} · gold = {GOLD} · at most {MAX_VIDEOS} video(s) · refetch after {REFETCH_DAYS} days · dry_run = {DRY_RUN}")

In [ ]:
# --- Food review videos from Gold (empty before the first classification run) ---
DIM_VIDEO = f"{GOLD}.dim_video"
video_ids = []
if spark.catalog.tableExists(DIM_VIDEO) and "video_type" in spark.table(DIM_VIDEO).columns:
    video_ids = [r.video_id for r in spark.sql(f"""
        SELECT video_id FROM {DIM_VIDEO}
        WHERE video_key <> -1 AND video_type IS NOT NULL AND is_food_review AND COALESCE(is_available, true)
    """).collect()]
print(f"{len(video_ids)} food review video(s) in {DIM_VIDEO}")

In [ ]:
# --- Fetch (never fails the job) ---
import json
import comments_task

try:
    result = comments_task.run_fetch_comments(video_ids, max_videos=MAX_VIDEOS, refetch_days=REFETCH_DAYS,
                                              dry_run=DRY_RUN)
except Exception as error:  # noqa: BLE001 - the daily run must go on with Bronze
    result = {"error": str(error)[:2000]}
    print(f"⚠️ fetch_comments failed, the daily job goes on: {error}")
dbutils.notebook.exit(json.dumps(result, ensure_ascii=False, default=str))